# NB10: Vibe Coding CEO Interface

Vibe coding is not "let the model do whatever it wants." In this course, vibe coding means a human acts as CEO of an AI workforce: describing intent in natural language, reviewing typed plans, changing direction mid-workflow, and approving or rejecting the final pull request summary.

## Management Principle

The human does not micromanage every token. The human manages goals, constraints, and approvals. The system converts conversational instructions into typed artifacts that agents can safely execute.

In [ ]:
from enum import Enum
from typing import Literal
from uuid import uuid4

from pydantic import BaseModel, ConfigDict, Field

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class AgentRole(str, Enum):
    CEO = "ceo"
    MANAGER = "manager"
    CODER = "coder"
    QA = "qa"
    SECURITY = "security"
    RELEASE = "release"

class ManagerInstruction(StrictModel):
    raw_text: str
    intent: Literal["create_project", "modify_constraint", "approve", "reject"]
    requested_by: AgentRole = AgentRole.CEO

class ProjectPlan(StrictModel):
    project_id: str = Field(default_factory=lambda: f"proj-{uuid4().hex[:6]}")
    goal: str
    api_style: Literal["REST", "GraphQL"] = "REST"
    database_choice: Literal["PostgreSQL", "MongoDB", "SQLite"]
    acceptance_criteria: list[str] = Field(min_length=1)
    risk_level: Literal["low", "medium", "high"]

class TeamCommitment(StrictModel):
    commitment_id: str = Field(default_factory=lambda: f"commit-{uuid4().hex[:6]}")
    owner: AgentRole
    key: str
    value: str
    reason: str

class HumanReviewDecision(StrictModel):
    status: Literal["approved", "rejected", "needs_changes"]
    reviewer: str
    notes: str

class CodePatch(StrictModel):
    files: dict[str, str]
    rationale: str

class TestResult(StrictModel):
    passed: bool
    log: str

class SecurityReview(StrictModel):
    approved: bool
    findings: list[str] = Field(default_factory=list)

class PullRequestSummary(StrictModel):
    title: str
    body: str
    database_choice: str
    approved_by_human: bool
    audit_trail: list[str]

class SharedMemory:
    def __init__(self):
        self.commitments: list[TeamCommitment] = []

    def add_commitment(self, commitment: TeamCommitment) -> None:
        self.commitments.append(commitment)

    def latest(self, key: str) -> TeamCommitment | None:
        matches = [item for item in self.commitments if item.key == key]
        return matches[-1] if matches else None

In [ ]:
def parse_manager_instruction(text: str) -> ManagerInstruction:
    lowered = text.lower()
    if "actually" in lowered or "instead" in lowered or "change" in lowered:
        intent = "modify_constraint"
    elif "approve" in lowered:
        intent = "approve"
    elif "reject" in lowered:
        intent = "reject"
    else:
        intent = "create_project"
    return ManagerInstruction(raw_text=text, intent=intent)

def mock_manager_llm_extract_plan(text: str) -> dict:
    """Teaching adapter: simulates an LLM translating natural language into a ProjectPlan candidate."""
    lowered = text.lower()
    if "database that doesn't exist" in lowered:
        database = "ImaginaryDB"  # Pydantic must reject this.
    elif "mongodb" in lowered:
        database = "MongoDB"
    elif "postgres" in lowered or "postgresql" in lowered:
        database = "PostgreSQL"
    else:
        database = "SQLite"

    return {
        "goal": text,
        "api_style": "GraphQL" if "graphql" in lowered else "REST",
        "database_choice": database,
        "acceptance_criteria": [
            "Expose user registration endpoint",
            "Expose login endpoint",
            "Persist users in the selected database",
            "Include tests and security review",
        ],
        "risk_level": "high" if "auth" in lowered or "secure" in lowered else "medium",
    }

def manager_create_plan(instruction: ManagerInstruction) -> ProjectPlan:
    return ProjectPlan.model_validate(
        mock_manager_llm_extract_plan(instruction.raw_text)
    )

def manager_apply_human_change(
    plan: ProjectPlan,
    instruction: ManagerInstruction,
    memory: SharedMemory,
) -> ProjectPlan:
    text = instruction.raw_text.lower()
    update = {}
    if "mongodb" in text:
        update["database_choice"] = "MongoDB"
        memory.add_commitment(
            TeamCommitment(
                owner=AgentRole.CEO,
                key="db",
                value="MongoDB",
                reason="Human CEO changed persistence requirement mid-workflow.",
            )
        )
    elif "postgres" in text:
        update["database_choice"] = "PostgreSQL"
        memory.add_commitment(
            TeamCommitment(
                owner=AgentRole.CEO,
                key="db",
                value="PostgreSQL",
                reason="Human CEO restored relational persistence requirement.",
            )
        )
    return plan.model_copy(update=update)

In [ ]:
def coder_agent(plan: ProjectPlan, memory: SharedMemory) -> CodePatch:
    db_commitment = memory.latest("db")
    database = db_commitment.value if db_commitment else plan.database_choice
    return CodePatch(
        files={
            "auth_api.py": (
                f"DATABASE = '{database}'\n"
                "def register_user(email, password):\n"
                "    return {'status': 'created', 'email': email}\n"
                "def login(email, password):\n"
                "    return {'token': 'mock-jwt'}\n"
            )
        },
        rationale=f"Implemented {plan.api_style} auth API using {database}.",
    )

def qa_agent(patch: CodePatch) -> TestResult:
    content = patch.files["auth_api.py"]
    required = ["register_user", "login", "DATABASE"]
    missing = [name for name in required if name not in content]
    return TestResult(
        passed=not missing,
        log="All auth API checks passed" if not missing else f"Missing: {missing}",
    )

def security_agent(plan: ProjectPlan, patch: CodePatch) -> SecurityReview:
    findings = []
    content = patch.files["auth_api.py"]
    if "password" not in content:
        findings.append("Authentication flow does not mention password handling.")
    if plan.risk_level == "high" and "mock-jwt" in content:
        findings.append("Teaching mock JWT must be replaced before production.")
    return SecurityReview(approved=True, findings=findings)

def release_manager(
    plan: ProjectPlan,
    patch: CodePatch,
    tests: TestResult,
    security: SecurityReview,
    human_decision: HumanReviewDecision,
    audit_trail: list[str],
) -> PullRequestSummary:
    return PullRequestSummary(
        title="Build user authentication API",
        body=(
            f"Goal: {plan.goal}\n"
            f"Database: {plan.database_choice}\n"
            f"Patch rationale: {patch.rationale}\n"
            f"QA: {tests.log}\n"
            f"Security findings: {security.findings}"
        ),
        database_choice=plan.database_choice,
        approved_by_human=human_decision.status == "approved",
        audit_trail=audit_trail,
    )

In [ ]:
def run_vibe_coding_session(scripted_inputs: list[str]) -> PullRequestSummary:
    memory = SharedMemory()
    audit_trail: list[str] = []

    create_instruction = parse_manager_instruction(scripted_inputs[0])
    plan = manager_create_plan(create_instruction)
    memory.add_commitment(
            TeamCommitment(
                owner=AgentRole.MANAGER,
                key="db",
                value=plan.database_choice,
                reason="Initial project plan from CEO instruction.",
            )
        )
    audit_trail.append(f"CEO request parsed into ProjectPlan using {plan.database_choice}")

    change_instruction = parse_manager_instruction(scripted_inputs[1])
    plan = manager_apply_human_change(plan, change_instruction, memory)
    audit_trail.append(f"CEO modified database choice to {plan.database_choice}")

    patch = coder_agent(plan, memory)
    tests = qa_agent(patch)
    security = security_agent(plan, patch)
    audit_trail.append(f"Coder produced patch: {patch.rationale}")
    audit_trail.append(f"QA result: {tests.log}")
    audit_trail.append(f"Security approved with findings: {security.findings}")

    human_decision = HumanReviewDecision(
        status="approved",
        reviewer="human-ceo",
        notes="Approved after confirming MongoDB requirement was honored.",
    )
    audit_trail.append(f"Human review status={human_decision.status}")

    return release_manager(
        plan=plan,
        patch=patch,
        tests=tests,
        security=security,
        human_decision=human_decision,
        audit_trail=audit_trail,
    )

summary = run_vibe_coding_session([
    "Build a REST API for user authentication with PostgreSQL",
    "Actually, use MongoDB instead",
])
print(summary.model_dump_json(indent=2))

assert summary.database_choice == "MongoDB"
assert summary.approved_by_human is True
assert any("modified database choice to MongoDB" in event for event in summary.audit_trail)

try:
    manager_create_plan(parse_manager_instruction("Build a REST API. Use a database that doesn't exist."))
    raise AssertionError("Invalid database should not validate.")
except Exception as exc:
    print(f"Validation blocked invalid CEO instruction: {type(exc).__name__}")

## CLI or Gradio Upgrade

The previous cell uses scripted inputs so the notebook is testable offline. To turn it into a real interface, wrap `run_vibe_coding_session` with either:

- a CLI loop that calls `input()` for each CEO instruction, or
- a Gradio text box that appends each instruction to session state.

Keep the same typed schemas. The interface changes, but the management contracts stay stable.

## 🧪 Exercises: The CEO's Steering Wheel

**The Story:** The CEO does not open `ProjectPlan` and edit JSON. The CEO says, "Build secure login," then later says, "Actually, use MongoDB and fix the security flaw." If that raw text flows straight to the Coder, the system becomes prompt soup. The manager must translate human intent into durable TeamLog state.

**Your Mission:** The CEO does not write Pydantic schemas. The CEO writes, "Make it faster and use MongoDB." The system must translate that chaos into governed state.

### Exercise 10.1: The Translation Layer

**Problem:** The CEO says, "Build a REST API for auth." The Coder needs a strict `ProjectPlan`.

**Task:** Write or extend `parse_manager_instruction()` and `manager_create_plan()`.

**Constraint:** Use a mock LLM adapter, or an optional live structured-output library, to extract `database_choice` and `api_style` into `ProjectPlan`.

**Validation:** If the CEO says, "Use a database that doesn't exist," schema validation must fail before the request reaches agents.

### Exercise 10.2: The Mid-Flight Pivot

**Problem:** The CEO changes direction after agents start: "Actually, use MongoDB."

**Task:** Implement `manager_apply_human_change()`.

**Logic:** The function must update `TeamCommitment(key="db", value="MongoDB")` in shared memory.

**The Nail:** The Coder must never see the raw text "Actually, use MongoDB." It only sees the updated TeamLog commitment.

### Exercise 10.3: The Veto Power

**Problem:** Agents may produce a patch the human should stop.

**Task:** Add `HumanReviewDecision(status: Literal["approved", "rejected", "needs_changes"])`.

**Logic:** If `rejected`, halt and log the decision. If `needs_changes`, route back to NB9's Debate Protocol.

## The Single Long Journey: From Whisper to Shipped Code

**Scenario:**
1. CEO types: "Build a secure login system."
2. Manager translates it into `ProjectPlan(risk_level="high")`.
3. NB9 generates a `WorkflowScaffold` that includes `SecurityReviewer`.
4. Coder writes the patch; Security Reviewer debates it and finds a flaw.
5. CEO sees the contention and types: "Fix the flaw and use MongoDB."
6. Manager updates `TeamCommitment(key="db", value="MongoDB")`.
7. Coder reads the new commitment, fixes the patch, and release waits for `HumanReviewDecision(status="approved")`.

**The Takeaway:** "Vibe Coding" is not magic. It is the rigorous translation of human intent into typed, governed TeamLog state, executed by an observable orchestration machine.